In [ ]:
!pip install -q gensim nltk scikit-learn pandas matplotlib

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gensim.downloader as api
from nltk.corpus import movie_reviews, stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)

for package in ['movie_reviews', 'punkt', 'punkt_tab', 'stopwords']:
    nltk.download(package, quiet=True)

In [ ]:
glove = api.load('glove-wiki-gigaword-300')
w2v = api.load('word2vec-google-news-300')

embeddings = {
    'GloVe (wiki-gigaword-300)': glove,
    'Word2Vec (Google News 300)': w2v,
}

pd.DataFrame([{'Модель': name, 'Слов в словаре': len(m.key_to_index), 'Размерность': m.vector_size}
              for name, m in embeddings.items()])

In [ ]:
def find_key(model, word):
    for candidate in (word, word.lower(), word.capitalize()):
        if candidate in model.key_to_index:
            return candidate
    return None


def similar(model, word, topn=8):
    key = find_key(model, word)
    if key is None:
        return 'нет в словаре'
    return ', '.join(f'{w} ({s:.2f})' for w, s in model.most_similar(key, topn=topn))


query_words = ['king', 'Minsk', 'pizza', 'computer', 'happy']

pd.DataFrame([{'Слово': word, 'Модель': name, 'Ближайшие слова': similar(model, word)}
              for word in query_words
              for name, model in embeddings.items()])

In [ ]:
def analogy(model, a, b, c, topn=5):
    keys = [find_key(model, w) for w in (a, b, c)]
    if None in keys:
        return []
    ka, kb, kc = keys
    return [w for w, _ in model.most_similar(positive=[kb, kc], negative=[ka], topn=topn)]


analogies = [
    ('France', 'Paris', 'Italy', 'Rome'),
    ('France', 'Paris', 'Belarus', 'Minsk'),
    ('Japan', 'Tokyo', 'Germany', 'Berlin'),
    ('Italy', 'pizza', 'Japan', 'sushi'),
    ('man', 'king', 'woman', 'queen'),
    ('walk', 'walked', 'swim', 'swam'),
    ('good', 'better', 'bad', 'worse'),
]

rows = []
for a, b, c, expected in analogies:
    row = {'Аналогия': f'{a} : {b} = {c} : ?', 'Ожидается': expected}
    for name, model in embeddings.items():
        top = analogy(model, a, b, c)
        top_lower = [w.lower() for w in top]
        row[f'{name}: top-1'] = top[0] if top else '-'
        row[f'{name}: top-5 содержит'] = expected.lower() in top_lower
    rows.append(row)

analogy_df = pd.DataFrame(rows)
display(analogy_df)

summary = []
for name in embeddings:
    top1 = np.mean([r[f'{name}: top-1'].lower() == r['Ожидается'].lower() for r in rows])
    top5 = np.mean([r[f'{name}: top-5 содержит'] for r in rows])
    summary.append({'Модель': name, 'Accuracy top-1': top1, 'Accuracy top-5': top5})
pd.DataFrame(summary)

In [ ]:
groups = {
    'страны': ['France', 'Germany', 'Italy', 'Japan', 'Belarus'],
    'столицы': ['Paris', 'Berlin', 'Rome', 'Tokyo', 'Minsk'],
    'животные': ['dog', 'cat', 'horse', 'lion', 'tiger'],
    'фрукты': ['apple', 'banana', 'orange', 'grape', 'lemon'],
    'технологии': ['computer', 'internet', 'software', 'keyboard', 'phone'],
}

words = [w for group in groups.values() for w in group]
colors = {group: plt.cm.tab10(i) for i, group in enumerate(groups)}
word_color = [colors[group] for group, ws in groups.items() for _ in ws]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for row, (name, model) in enumerate(embeddings.items()):
    vectors = np.array([model[find_key(model, w)] for w in words])
    projections = {
        'PCA': PCA(n_components=2, random_state=42).fit_transform(vectors),
        't-SNE': TSNE(n_components=2, perplexity=5, init='pca',
                      learning_rate='auto', random_state=42).fit_transform(vectors),
    }
    for col, (method, points) in enumerate(projections.items()):
        ax = axes[row, col]
        ax.scatter(points[:, 0], points[:, 1], c=word_color, s=80)
        for (x, y), word in zip(points, words):
            ax.annotate(word, (x, y), textcoords='offset points', xytext=(5, 5), fontsize=11)
        ax.set_title(f'{name}: {method}')
        ax.grid(True, alpha=0.3)

handles = [plt.Line2D([0], [0], marker='o', linestyle='', color=color, label=group)
           for group, color in colors.items()]
fig.legend(handles=handles, loc='lower center', ncol=len(groups), fontsize=12)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

In [ ]:
docs = [(movie_reviews.raw(file_id), 1 if category == 'pos' else 0)
        for category in movie_reviews.categories()
        for file_id in movie_reviews.fileids(category)]

df = pd.DataFrame(docs, columns=['text', 'label'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

negations = {'no', 'nor', 'not', 'don', 'didn', 'doesn', 'isn', 'wasn', 'aren', 'weren',
             'won', 'wouldn', 'couldn', 'shouldn', 'hasn', 'haven', 'hadn'}
stop_words = set(stopwords.words('english')) - negations


def tokenize(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
    return [t for t in word_tokenize(text) if len(t) > 1 and t not in stop_words]


df['tokens'] = df['text'].apply(tokenize)
df['clean'] = df['tokens'].apply(' '.join)

print(df.shape)
print(df['label'].value_counts())
df[['clean', 'label']].head()

In [ ]:
def doc_vector(tokens, model):
    vectors = [model[t] for t in tokens if t in model.key_to_index]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


coverage = []
for name, model in embeddings.items():
    total = sum(len(t) for t in df['tokens'])
    known = sum(sum(w in model.key_to_index for w in t) for t in df['tokens'])
    empty = sum(not any(w in model.key_to_index for w in t) for t in df['tokens'])
    coverage.append({'Модель': name, 'Покрытие токенов': known / total, 'Документов без известных слов': empty})
pd.DataFrame(coverage)

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df['label'])
y_train, y_test = df.loc[train_idx, 'label'], df.loc[test_idx, 'label']

tfidf = TfidfVectorizer(max_features=10000, min_df=2, ngram_range=(1, 2))

features = {
    'TF-IDF (лаба 10)': (tfidf.fit_transform(df.loc[train_idx, 'clean']),
                         tfidf.transform(df.loc[test_idx, 'clean']), False),
}

for name, model in embeddings.items():
    matrix = np.vstack(df['tokens'].apply(lambda t: doc_vector(t, model)).values)
    features[name] = (matrix[train_idx], matrix[test_idx], True)


def make_models(scale):
    wrap = (lambda m: make_pipeline(StandardScaler(), m)) if scale else (lambda m: m)
    return {
        'Logistic Regression': wrap(LogisticRegression(max_iter=2000)),
        'SVM': wrap(LinearSVC(max_iter=10000, random_state=42)),
        'Random Forest': wrap(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    }


results = []
for representation, (X_tr, X_te, scale) in features.items():
    for model_name, model in make_models(scale).items():
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        results.append({
            'Представление': representation,
            'Модель': model_name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1': f1_score(y_test, y_pred),
        })

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_df

In [ ]:
results_df.pivot(index='Модель', columns='Представление', values='Accuracy').round(4)